In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
print("✅ Фикс OpenMP применён")

✅ Фикс OpenMP применён


In [ ]:
import os
import glob

DATA_DIR = r'C:\Users\Shever\jupyter\ruda\data'

if not os.path.exists(DATA_DIR):
    print(f"❌ Папка {DATA_DIR} не найдена!")
    print("Проверь путь или создай папку data.")
    exit()

print(f"✅ Папка data найдена: {DATA_DIR}")
print("=" * 60)

# ============================================
# 3. СЧИТАЕМ КАРТИНКИ ПО КЛАССАМ (ЧАСТЬ 1)
# ============================================
print("\n📊 Фото руд по сортам. ч1:")

base1 = os.path.join(DATA_DIR, 'Фото руд по сортам. ч1')
classes_1 = ['Оталькованные руды', 'Рядовые руды', 'Труднообогатимые руды']
class_counts_1 = {}

for cls in classes_1:
    folder_path = os.path.join(base1, cls)
    if os.path.exists(folder_path):
        images = glob.glob(os.path.join(folder_path, '*.[Jj][Pp][Gg]'))
        class_counts_1[cls] = len(images)
        print(f"  {cls}: {len(images)} картинок")
    else:
        print(f"  ❌ Папка '{cls}' не найдена")
        class_counts_1[cls] = 0

# ============================================
# 4. СЧИТАЕМ КАРТИНКИ ПО КЛАССАМ (ЧАСТЬ 2)
# ============================================
print("\n📊 Фото руд по сортам. ч2:")

base2 = os.path.join(DATA_DIR, 'Фото руд по сортам. ч2')
classes_2 = ['оталькованные', 'рядовые', 'тонкие']
class_counts_2 = {}

for cls in classes_2:
    folder_path = os.path.join(base2, cls)
    if os.path.exists(folder_path):
        images = glob.glob(os.path.join(folder_path, '*.[Jj][Pp][Gg]'))
        class_counts_2[cls] = len(images)
        print(f"  {cls}: {len(images)} картинок")
    else:
        print(f"  ❌ Папка '{cls}' не найдена")
        class_counts_2[cls] = 0

# ============================================
# 5. ПРОВЕРЯЕМ ПАПКУ С МАСКАМИ
# ============================================
print("\n🎭 Проверка масок...")

mask_base = os.path.join(
    DATA_DIR,
    'Фото руд по сортам. ч1',
    'Оталькованные руды',
    'Области оталькования'
)

if os.path.exists(mask_base):
    masks = glob.glob(os.path.join(mask_base, '*.JPG')) + glob.glob(os.path.join(mask_base, '*.JPG'))
    print(f"  Найдено масок: {len(masks)}")
    
    if masks:
        mask_names = set([os.path.splitext(os.path.basename(m))[0] for m in masks])
        
        ore_folder = os.path.join(DATA_DIR, 'Фото руд по сортам. ч1', 'Оталькованные руды')
        ores = glob.glob(os.path.join(ore_folder, '*.[Jj][Pp][Gg]'))
        ore_names = set([os.path.splitext(os.path.basename(o))[0] for o in ores])
        
        common = mask_names & ore_names
        only_in_mask = mask_names - ore_names
        only_in_ore = ore_names - mask_names
        
        print(f"  Маски + картинки: {len(common)} пар")
        if only_in_mask:
            print(f"  ⚠️ Есть маски без картинок: {list(only_in_mask)[:5]}")
        if only_in_ore:
            print(f"  ⚠️ Есть картинки без масок: {list(only_in_ore)[:5]}")
else:
    print("  ❌ Папка 'Области оталькования' не найдена")

# ============================================
# 6. ПРОВЕРЯЕМ ПАНОРАМЫ
# ============================================
print("\n🌄 Панорамы...")

panorama_path = os.path.join(DATA_DIR, 'Панорамы')
if os.path.exists(panorama_path):
    panoramas = glob.glob(os.path.join(panorama_path, '*.[Jj][Pp][Gg]'))
    print(f"  Найдено панорам: {len(panoramas)}")
else:
    print("  ❌ Папка 'Панорамы' не найдена")

# ============================================
# 7. ИТОГОВАЯ СТАТИСТИКА
# ============================================
print("\n" + "=" * 60)
print("📋 ИТОГОВАЯ СТАТИСТИКА")
print("=" * 60)

total = 0
print("\nЧасть 1:")
for cls, count in class_counts_1.items():
    print(f"  {cls}: {count}")
    total += count

print("\nЧасть 2:")
for cls, count in class_counts_2.items():
    print(f"  {cls}: {count}")
    total += count

print(f"\n📊 ВСЕГО КАРТИНОК: {total}")

print("\n" + "=" * 60)
print("✅ РАЗВЕДКА ЗАВЕРШЕНА!")

In [ ]:
# ============================================
# ШАГ 1: ПОДГОТОВКА ДАННЫХ
# ============================================

import os
import cv2
import numpy as np
import glob
from sklearn.model_selection import train_test_split
import shutil

# Пути
DATA_DIR = r'C:\Users\Shever\jupyter\ruda\data'
OUTPUT_DIR = r'C:\Users\Shever\jupyter\ruda\processed_data'

# Создаём структуру папок для обработанных данных
for split in ['train', 'val', 'test']:
    for cls in ['otalkovannye', 'ryadovye', 'trudnoobogatimye']:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'masks_talc'), exist_ok=True)

print("✅ Структура папок создана")

# ============================================
# 1.1 ИЗВЛЕЧЕНИЕ МАСОК ТАЛЬКА (выделение синего контура)
# ============================================

def extract_talc_mask(mask_path):
    """
    На входе: изображение с синей линией (маска талька от эксперта)
    На выходе: бинарная маска (255 = тальк, 0 = фон)
    """
    img = cv2.imread(mask_path)
    if img is None:
        print(f"❌ Не могу прочитать: {mask_path}")
        return None
    
    # Переводим в HSV для выделения синего цвета
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Диапазон синего цвета
    lower_blue = np.array([90, 50, 50])
    upper_blue = np.array([140, 255, 255])
    
    mask = cv2.inRange(hsv, lower_blue, upper_blue)
    
    # Морфология: закрываем дырки внутри контура
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Заполняем внутренность контура
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(mask)
    cv2.drawContours(filled_mask, contours, -1, 255, thickness=cv2.FILLED)
    
    return filled_mask


# Обрабатываем все маски
mask_folder = os.path.join(DATA_DIR, 'Фото руд по сортам. ч1', 'Оталькованные руды', 'Области оталькования')
mask_files = glob.glob(os.path.join(mask_folder, '*.JPG')) + glob.glob(os.path.join(mask_folder, '*.jpg'))

print(f"\n🎭 Обработка масок талька: {len(mask_files)} файлов")

processed_masks = {}
for mask_path in mask_files:
    mask = extract_talc_mask(mask_path)
    if mask is not None:
        # Имя файла без расширения
        fname = os.path.splitext(os.path.basename(mask_path))[0]
        processed_masks[fname] = mask

print(f"✅ Успешно обработано масок: {len(processed_masks)}")

# ============================================
# 1.2 КОПИРУЕМ ЧАСТЬ 1 В TEST (с масками)
# ============================================

base1 = os.path.join(DATA_DIR, 'Фото руд по сортам. ч1')
ore_folder_1 = os.path.join(base1, 'Оталькованные руды')

# Копируем оталькованные из части 1
ore_files = glob.glob(os.path.join(ore_folder_1, '*.[Jj][Pp][Gg]'))
ore_files = [f for f in ore_files if 'Области оталькования' not in f]

for img_path in ore_files:
    fname = os.path.splitext(os.path.basename(img_path))[0]
    dst_img = os.path.join(OUTPUT_DIR, 'test', 'otalkovannye', os.path.basename(img_path))
    shutil.copy2(img_path, dst_img)
    
    # Если есть парная маска — сохраняем
    if fname in processed_masks:
        dst_mask = os.path.join(OUTPUT_DIR, 'test', 'masks_talc', fname + '.png')
        cv2.imwrite(dst_mask, processed_masks[fname])

# Копируем рядовые и труднообогатимые
for cls, folder_name in [('ryadovye', 'Рядовые руды'), ('trudnoobogatimye', 'Труднообогатимые руды')]:
    src_folder = os.path.join(base1, folder_name)
    dst_folder = os.path.join(OUTPUT_DIR, 'test', cls)
    for img_path in glob.glob(os.path.join(src_folder, '*.[Jj][Pp][Gg]')):
        shutil.copy2(img_path, os.path.join(dst_folder, os.path.basename(img_path)))

print("✅ Часть 1 скопирована в test")

# ============================================
# 1.3 РАЗДЕЛЯЕМ ЧАСТЬ 2 НА TRAIN/VAL
# ============================================

base2 = os.path.join(DATA_DIR, 'Фото руд по сортам. ч2')
class_mapping = {
    'оталькованные': 'otalkovannye',
    'рядовые': 'ryadovye',
    'тонкие': 'trudnoobogatimye'
}

# Собираем все файлы части 2
part2_files = []
part2_labels = []

for folder_name, cls in class_mapping.items():
    folder_path = os.path.join(base2, folder_name)
    if os.path.exists(folder_path):
        files = glob.glob(os.path.join(folder_path, '*.[Jj][Pp][Gg]'))
        for f in files:
            part2_files.append(f)
            part2_labels.append(cls)

print(f"\n📊 Часть 2: {len(part2_files)} изображений")

# Стратифицированное разделение 85/15
train_files, val_files, train_labels, val_labels = train_test_split(
    part2_files, part2_labels,
    test_size=0.15,
    stratify=part2_labels,
    random_state=42
)

print(f"  Train: {len(train_files)}")
print(f"  Val: {len(val_files)}")

# Копируем train
for img_path, label in zip(train_files, train_labels):
    dst = os.path.join(OUTPUT_DIR, 'train', label, os.path.basename(img_path))
    shutil.copy2(img_path, dst)

# Копируем val
for img_path, label in zip(val_files, val_labels):
    dst = os.path.join(OUTPUT_DIR, 'val', label, os.path.basename(img_path))
    shutil.copy2(img_path, dst)

print("✅ Часть 2 разделена и скопирована")

# ============================================
# 1.4 СТАТИСТИКА
# ============================================

print("\n" + "=" * 60)
print("📋 ИТОГОВАЯ СТАТИСТИКА ДАТАСЕТА")
print("=" * 60)

for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    total_split = 0
    for cls in ['otalkovannye', 'ryadovye', 'trudnoobogatimye']:
        folder = os.path.join(OUTPUT_DIR, split, cls)
        count = len(glob.glob(os.path.join(folder, '*.[Jj][Pp][Gg]')))
        print(f"  {cls}: {count}")
        total_split += count
    
    masks_count = len(glob.glob(os.path.join(OUTPUT_DIR, split, 'masks_talc', '*.png')))
    if masks_count > 0:
        print(f"  маски талька: {masks_count}")
    print(f"  ВСЕГО: {total_split}")

print("\n" + "=" * 60)
print("✅ ШАГ 1 ЗАВЕРШЁН!")
print(f"Данные в: {OUTPUT_DIR}")

In [ ]:
# Тренировка основной CNN
import torch
torch.backends.cudnn.benchmark = True

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np
import os
from PIL import Image, ImageOps
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('PIL').setLevel(logging.WARNING)

# ============================================
# КОНСТАНТЫ
# ============================================

CLASSES = ['otalkovannye', 'ryadovye', 'trudnoobogatimye']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
IMAGE_SIZE = 456
PROCESSED_DIR = r'C:\Users\Shever\jupyter\ruda\processed_data'
MODEL_DIR = r'C:\Users\Shever\jupyter\ruda\models'
BATCH_SIZE = 32
SEED = 42

# ============================================
# DATASET
# ============================================

class OreDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.root_dir = os.path.join(root_dir, split)
        self.images = []
        self.labels = []
        
        for cls in CLASSES:
            cls_folder = os.path.join(self.root_dir, cls)
            if not os.path.exists(cls_folder):
                continue
            for fname in os.listdir(cls_folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    self.images.append(os.path.join(cls_folder, fname))
                    self.labels.append(CLASS_TO_IDX[cls])
        
        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=25),
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                transforms.RandomErasing(p=0.15, scale=(0.02, 0.1)),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            img = ImageOps.autocontrast(img, cutoff=2)
        except:
            return self.__getitem__((idx + 1) % len(self))
        img = self.transform(img)
        return img, label

# ============================================
# МОДЕЛЬ
# ============================================

class OreClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = efficientnet_b5(weights=EfficientNet_B5_Weights.IMAGENET1K_V1)
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)
    
    def unfreeze_backbone(self, pct=0.4):
        params = list(self.backbone.parameters())
        total = len(params)
        for i, param in enumerate(params):
            if i >= total * (1 - pct):
                param.requires_grad = True

# ============================================
# ФУНКЦИИ ОБУЧЕНИЯ
# ============================================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_true = [], []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_true.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_true, all_preds, average='macro')
    return avg_loss, f1

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_true, all_probs = [], [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_true, all_preds, average='macro')
    try:
        auc = roc_auc_score(all_true, all_probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    return avg_loss, f1, auc

# ============================================
# MAIN
# ============================================

if __name__ == '__main__':
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"🖥️  Устройство: {DEVICE}")
    print(f"📐 Размер изображений: {IMAGE_SIZE}px")
    print(f"🧠 Модель: EfficientNet-B5")
    
    # Загрузка данных
    train_dataset = OreDataset(PROCESSED_DIR, 'train')
    val_dataset = OreDataset(PROCESSED_DIR, 'val')
    test_dataset = OreDataset(PROCESSED_DIR, 'test')
    
    print(f"\n📊 Загружено: Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    
    # Балансировка классов
    labels = train_dataset.labels
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    print(f"⚖️  Веса классов: {class_weights}")
    
    # Модель
    model = OreClassifier(num_classes=3).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n🔧 Параметров всего: {total_params:,}, Обучаемых: {trainable_params:,}")
    
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion_stage1 = nn.CrossEntropyLoss(weight=class_weights_tensor)
    criterion_stage2 = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    best_model_path = os.path.join(MODEL_DIR, 'classifier_b5_final.pth')
    best_val_f1 = 0
    
    # ЭТАП 1
    print("\n" + "=" * 70)
    print("ЭТАП 1: Обучение классификатора (backbone заморожен)")
    print("=" * 70)
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=12, eta_min=1e-6)
    
    for epoch in range(12):
        train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion_stage1, DEVICE)
        val_loss, val_f1, val_auc = val_epoch(model, val_loader, criterion_stage1, DEVICE)
        scheduler.step()
        
        gap = train_f1 - val_f1
        status = "💾" if val_f1 > best_val_f1 else ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
        
        print(f"Epoch {epoch+1:2d}/12 | Tr F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | AUC: {val_auc:.4f} | Gap: {gap:+.4f} {status}")
    
    print(f"\n✅ Этап 1 завершён. Лучший Val F1: {best_val_f1:.4f}")
    
    # ЭТАП 2
    print("\n" + "=" * 70)
    print("ЭТАП 2: Тонкая настройка (разморожено 40% backbone)")
    print("=" * 70)
    
    model.unfreeze_backbone(pct=0.4)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🔧 Обучаемых параметров: {trainable_params:,}")
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-7)
    
    patience = 8
    epochs_no_improve = 0
    
    for epoch in range(25):
        train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion_stage2, DEVICE)
        val_loss, val_f1, val_auc = val_epoch(model, val_loader, criterion_stage2, DEVICE)
        scheduler.step()
        
        gap = train_f1 - val_f1
        status = ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            status = "💾"
        else:
            epochs_no_improve += 1
        
        print(f"Epoch {epoch+1:2d}/25 | Tr F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | AUC: {val_auc:.4f} | Gap: {gap:+.4f} {status}")
        
        if epochs_no_improve >= patience:
            print(f"  ⏹️ Early stopping на эпохе {epoch+1}")
            break
    
    print(f"\n✅ Обучение завершено. Лучший Val F1: {best_val_f1:.4f}")
    print(f"✅ Модель сохранена: {best_model_path}")